# 03 -- Stem Separation + Embedding (Stretch tier)

Splits each song into vocal/drums/bass/instrumental stems (Demucs), then embeds
each stem independently through its own facet (`VocalFacet`, `DrumsFacet`,
`BassFacet`, `InstrumentalFacet` -- each is just `SoundFacet`/CLAP run on
isolated audio, see `sonic_explorer/facets/stems.py`). Reuses the real
`sonic_explorer` package, same as every other notebook here -- no
notebook-local reimplementation.

**Requires a GPU runtime**: Runtime -> Change runtime type -> T4 GPU (or
better). Demucs is a heavy neural model; CPU-only would take hours-to-overnight
for the full library.

**Honesty check before the full run**: the Demucs integration
(`pipeline/separation.py`) was written from documented API knowledge, not
empirically validated -- this project's dev machine has no GPU and demucs/torch
aren't installed there. Section 4 below is a deliberate smoke test: separate
one song, *listen* to the four stems, confirm they're actually isolated (vocals
sound like vocals, drums sound like drums) before committing to a full
1400-song run. If something's off, this is the cell to debug against.

**Output** (in `MyDrive/SonicExplorer/artifacts/`):
- `vocal.index`, `drums.index`, `bass.index`, `instrumental.index` -- FAISS
  indexes, one per stem facet
- `sonic_explorer.db` -- same DB as before, now with embedding_status rows for
  the four new facets too

## 1. Clone the repo and install sonic_explorer (with the `[colab]` extra)

Now includes `demucs` + `torchaudio` alongside torch/transformers/librosa.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/oyoai/sonic-explorer.git'
REPO_DIR = '/content/sonic-explorer'


def run(cmd):
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Command failed (exit {result.returncode}): {" ".join(cmd)}')


if os.path.exists(f'{REPO_DIR}/.git'):
    run(['git', '-C', REPO_DIR, 'pull'])
else:
    run(['git', 'clone', REPO_URL, REPO_DIR])

run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[colab]'])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('sonic_explorer installed from', REPO_DIR)

## 2. Mount Drive and set up paths

Same Drive-mounted DB + artifacts folder every other notebook uses -- the
1400 songs and their sound/harmony embeddings should already be there from
notebooks 01/02.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/SonicExplorer')
CURATED_DIR = DRIVE_ROOT / 'fma_curated'
ARTIFACTS_DIR = DRIVE_ROOT / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = ARTIFACTS_DIR / 'sonic_explorer.db'

print('Curated audio:', CURATED_DIR)
print('Artifacts (DB + indexes):', ARTIFACTS_DIR)

## 3. Load repositories + build the manifest from the DB

Unlike notebook 02, this doesn't need `curated_tracks.csv` -- every song
should already exist in the DB (added during the sound-facet batch job), so
the manifest is just `track_id` + `relative_path` built straight from
`song_repo.list_songs()`, same pattern as the local structure/harmony
pipeline scripts. `run_batch_stem_embedding` skips any track_id not already
in the DB rather than creating one.

In [ ]:
import pandas as pd

from sonic_explorer.repository.db import init_db
from sonic_explorer.repository.song_repository import SongRepository
from sonic_explorer.repository.embedding_repository import EmbeddingRepository
from sonic_explorer.facets.stems import VocalFacet, DrumsFacet, BassFacet, InstrumentalFacet

conn = init_db(DB_PATH)
song_repo = SongRepository(conn)
embedding_repo = EmbeddingRepository(conn, artifacts_dir=ARTIFACTS_DIR)

STEM_FACETS = {
    'vocal': VocalFacet(),
    'drums': DrumsFacet(),
    'bass': BassFacet(),
    'instrumental': InstrumentalFacet(),
}
for name in STEM_FACETS:
    embedding_repo.load_index(name)
    print(f'Resuming {name} index size:', embedding_repo.index_size(name))

songs = song_repo.list_songs()
manifest = pd.DataFrame([
    {'track_id': s.fma_track_id, 'relative_path': f'{s.fma_track_id}.mp3'} for s in songs
])
print(f'{len(manifest)} songs to process')

## 4. Smoke test -- separate ONE song and listen

Do this before the full run. If the stems aren't actually isolated (e.g. the
"vocal" stem still has a full drum kit in it), something's wrong with the
separation wrapper and it's much cheaper to find out now than after an hour
of GPU time.

In [ ]:
import random
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

from sonic_explorer.config import CLAP_SR
from sonic_explorer.pipeline.separation import separate_stems

sample = random.choice(songs)
print(f'Smoke-testing separation on: "{sample.title}" by {sample.artist}')

import librosa
audio, sr = librosa.load(str(CURATED_DIR / f'{sample.fma_track_id}.mp3'), sr=CLAP_SR, mono=True)
# just the first 20s -- plenty to judge separation quality, much faster than the whole song
clip = audio[: 20 * sr]

stems = separate_stems(clip, sr)
for name, stem_audio in stems.items():
    print(f'-- {name} --')
    display(Audio(stem_audio, rate=sr))

## 5. Run the batch pipeline

Same checkpointing discipline as every other batch job here: a segment is
only ever marked `'done'` in the DB after its vector is durably saved to the
FAISS index on disk, and a single song's separation/embedding failure is
isolated (reported, not fatal) rather than taking down the whole run --
Demucs failing on one malformed file shouldn't lose progress on the other
1399.

In [ ]:
from sonic_explorer.pipeline.embed_stems import run_batch_stem_embedding

def log_progress(done, total):
    sizes = {name: embedding_repo.index_size(name) for name in STEM_FACETS}
    print(f'[{done}/{total}] checkpoint -- index sizes: {sizes}')

def log_error(track_id, exc):
    print(f'  WARNING: track {track_id} failed ({type(exc).__name__}: {exc}) -- skipped, will retry next run')

failed = run_batch_stem_embedding(
    manifest_df=manifest,
    audio_dir=CURATED_DIR,
    song_repo=song_repo,
    embedding_repo=embedding_repo,
    stem_facets=STEM_FACETS,
    checkpoint_every=25,
    on_checkpoint=log_progress,
    on_error=log_error,
)

print('Batch stem embedding complete.')
for name in STEM_FACETS:
    print(f'Final {name} index size:', embedding_repo.index_size(name))
if failed:
    print(f'{len(failed)} track(s) failed and were skipped: {failed}')

## 6. Sanity check -- does a stem facet actually return something sensible?

Same pattern as notebook 02's check: pull a segment's vocal-facet vector back
out (no re-separation needed) and look at its nearest neighbors in other
songs, excluding the query's own song.

In [ ]:
sample_song = random.choice(song_repo.list_songs())
sample_song = song_repo.get_song(sample_song.id)
query_seg = sample_song.segments[len(sample_song.segments) // 2]

if embedding_repo.status(query_seg.id, 'vocal') == 'done':
    query_vec = embedding_repo.get_vector('vocal', query_seg.id)
    raw_matches = embedding_repo.search('vocal', query_vec, k=20)
    other_song_matches = [
        (seg_id, score) for seg_id, score in raw_matches
        if song_repo.get_segment(seg_id).song_id != sample_song.id
    ][:6]

    print(f'Query (vocal facet): "{sample_song.title}" by {sample_song.artist}\n')
    for seg_id, score in other_song_matches:
        seg = song_repo.get_segment(seg_id)
        match_song = song_repo.get_song(seg.song_id)
        print(f'  {score:.3f}  "{match_song.title}" by {match_song.artist} ({match_song.genre_top})')
else:
    print(f'"{sample_song.title}" has no vocal-facet vector yet -- pick another sample or re-run the batch cell.')

## Done

`MyDrive/SonicExplorer/artifacts/` now has `vocal.index`, `drums.index`,
`bass.index`, `instrumental.index` alongside `sound.index` and
`harmony.index`. Sync `sonic_explorer.db` and the four new `.index` files
down to this repo's local `data/artifacts/` the same way notebook 02's output
was synced, then the Explore page's facet toggle/blend will pick them up
automatically -- no code changes needed on the local/deployed side, the
registry already has these facets wired in.